To run this notebook, you will need a `.env` file at the root of the project. It should contain the following keys:
```
LLM_SERVICE=OpenAI
OPENAI_API_KEY=...
OPENAI_ENDPOINT=...
OPENAI_DEPLOYMENT_NAME=...
OPENAI_API_VERSION=...
```

The first step is to load the libraries we'll be using. I am importing `pandas` for basic data manipulation, and a package called [discovery_utils](https://github.com/nestauk/discovery_utils) that Karlis made. It contains various functions that we have used and reused across Discovery projects, including functions for extracting structured information using LLMs. You can read more about how the `llm` module works [here](https://github.com/nestauk/discovery_utils/wiki/Checking-data-with-LLM).

In [1]:
import pandas as pd
from discovery_utils.utils.llm import batch_check

from discovery_heat_pump_futures import PROJECT_DIR

Karlis has created two datasets for this project: one containing research abstracts from [OpenAlex](https://openalex.org/), and one from [Google Patents](https://patents.google.com/).

In [2]:
openalex_df = pd.read_csv(
    "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_openalex.csv"
)
patents_df = pd.read_json(
    "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_patents.json",
    lines=True,
)

/tmp/ipykernel_7240/865600020.py:1: DtypeWarning: Columns (52) have mixed types. Specify dtype option on import or set low_memory=False.
  openalex_df = pd.read_csv(


The patents dataset looks like this:

In [3]:
patents_df.head()

,publication_number,application_number,family_id,url,filing_date,grant_date,publication_date,priority_date,title,title_translated,abstract,abstract_translated,top_terms,country_code,inventor_harmonized,assignee_harmonized,cpc
0,WO-2024184245-A1,EP-2024055431-W,85462233,https://patents.google.com/patent/WO2024184245A1,20240301,0,20240912,20230303,Heat source unit of an air heat pump,False,"Heat source unit (1) of an air heat pump, havi...",False,"[fan, louvers, heat source, source unit, cente...",WO,"[{'name': 'HEINTZ NICOLAS', 'country_code': 'B...","[{'name': 'DAIKIN EUROPE NV', 'country_code': ...","[{'code': 'F24F13/082', 'inventive': False, 'f..."
1,US-11994321-B2,US-202017420370-A,71406811,https://patents.google.com/patent/US11994321B2,20200103,20240528,20240528,20190103,High performance compressors and vapor compres...,False,The present disclosure relates to a new breed ...,False,"[compressor, pressure, motor, refrigerant, pum...",US,"[{'name': 'LEE KANG P', 'country_code': 'US'},...","[{'name': 'ASPEN COMPRESSOR LLC', 'country_cod...","[{'code': 'F25B31/026', 'inventive': True, 'fi..."
2,WO-2022023086-A1,EP-2021070024-W,71846293,https://patents.google.com/patent/WO2022023086A1,20210716,0,20220203,20200729,A heating system,False,A heating system (1) has a turbine (20) for bu...,False,"[flue gas, heat exchanger, air, wshp, water, h...",WO,"[{'name': 'FLYNN LIAM', 'country_code': 'IE'},...","[{'name': 'ENERGY SERVICES LTD', 'country_code...","[{'code': 'Y02B30/18', 'inventive': False, 'fi..."
3,CN-211854451-U,CN-202020379578-U,73135275,https://patents.google.com/patent/CN211854451U,20200323,20201103,20201103,20200323,Cold and hot water supply system for poultry s...,False,The utility model provides a poultry is slaugh...,False,"[hot water, cold, water pipe, supply system, p...",CN,"[{'name': 'WU TAN'EN', 'country_code': ''}]",[{'name': 'TAIZHOU HONGYE ENVIRONMENTAL TECH C...,[]
4,WO-2024162444-A1,JP-2024003342-W,91030893,https://patents.google.com/patent/WO2024162444A1,20240201,0,20240808,20230202,Heat pump device,False,A heat pump device according to one embodiment...,False,"[discharge, temperature, compressor, pressure,...",WO,"[{'name': 'SAGAWA KENTARO', 'country_code': 'J...","[{'name': 'FUJITSU GENERAL LTD', 'country_code...","[{'code': 'F25B1/00', 'inventive': True, 'firs..."


Below, we specify a helper function that can also be used on the OpenAlex data to concatenate the titles and abstracts of individual patents/abstracts. This new `"title_abstract"` field will be the input to `LLMProcessor`.

In [4]:
def format_title_abstract(
    df: pd.DataFrame, title: str = "title", abstract: str = "abstract"
) -> pd.DataFrame:
    """Format the title and abstract for LLM input.

    Args:
        df (pd.DataFrame): DataFrame containing the title and abstract columns.
        title (str): Name of the title column.
        abstract (str): Name of the abstract column.

    Returns:
        pd.DataFrame: DataFrame with a new column 'title_abstract'
            containing formatted text.
    """
    df["title_abstract"] = (
        "TITLE: "
        + df[title].str.lower().fillna("")
        + " ABSTRACT: "
        + df[abstract].str.lower().fillna("")
    )
    return df


patents_df = format_title_abstract(patents_df, title="title", abstract="abstract")
patents_df.head()

,publication_number,application_number,family_id,url,filing_date,grant_date,publication_date,priority_date,title,title_translated,abstract,abstract_translated,top_terms,country_code,inventor_harmonized,assignee_harmonized,cpc,title_abstract
0,WO-2024184245-A1,EP-2024055431-W,85462233,https://patents.google.com/patent/WO2024184245A1,20240301,0,20240912,20230303,Heat source unit of an air heat pump,False,"Heat source unit (1) of an air heat pump, havi...",False,"[fan, louvers, heat source, source unit, cente...",WO,"[{'name': 'HEINTZ NICOLAS', 'country_code': 'B...","[{'name': 'DAIKIN EUROPE NV', 'country_code': ...","[{'code': 'F24F13/082', 'inventive': False, 'f...",TITLE: heat source unit of an air heat pump AB...
1,US-11994321-B2,US-202017420370-A,71406811,https://patents.google.com/patent/US11994321B2,20200103,20240528,20240528,20190103,High performance compressors and vapor compres...,False,The present disclosure relates to a new breed ...,False,"[compressor, pressure, motor, refrigerant, pum...",US,"[{'name': 'LEE KANG P', 'country_code': 'US'},...","[{'name': 'ASPEN COMPRESSOR LLC', 'country_cod...","[{'code': 'F25B31/026', 'inventive': True, 'fi...",TITLE: high performance compressors and vapor ...
2,WO-2022023086-A1,EP-2021070024-W,71846293,https://patents.google.com/patent/WO2022023086A1,20210716,0,20220203,20200729,A heating system,False,A heating system (1) has a turbine (20) for bu...,False,"[flue gas, heat exchanger, air, wshp, water, h...",WO,"[{'name': 'FLYNN LIAM', 'country_code': 'IE'},...","[{'name': 'ENERGY SERVICES LTD', 'country_code...","[{'code': 'Y02B30/18', 'inventive': False, 'fi...",TITLE: a heating system ABSTRACT: a heating sy...
3,CN-211854451-U,CN-202020379578-U,73135275,https://patents.google.com/patent/CN211854451U,20200323,20201103,20201103,20200323,Cold and hot water supply system for poultry s...,False,The utility model provides a poultry is slaugh...,False,"[hot water, cold, water pipe, supply system, p...",CN,"[{'name': 'WU TAN'EN', 'country_code': ''}]",[{'name': 'TAIZHOU HONGYE ENVIRONMENTAL TECH C...,[],TITLE: cold and hot water supply system for po...
4,WO-2024162444-A1,JP-2024003342-W,91030893,https://patents.google.com/patent/WO2024162444A1,20240201,0,20240808,20230202,Heat pump device,False,A heat pump device according to one embodiment...,False,"[discharge, temperature, compressor, pressure,...",WO,"[{'name': 'SAGAWA KENTARO', 'country_code': 'J...","[{'name': 'FUJITSU GENERAL LTD', 'country_code...","[{'code': 'F25B1/00', 'inventive': True, 'firs...",TITLE: heat pump device ABSTRACT: a heat pump ...


In [5]:
# to show what the formatted column looks like
patents_df["title_abstract"].values[0]

'TITLE: heat source unit of an air heat pump ABSTRACT: heat source unit (1) of an air heat pump, having a casing (10), a fan (22) accommodated in the casing (10) and being rotatable about a center axis (30), a bell mouth (32) having an opening (32), and a grille (40, 42) covering the opening (32). the grille (40, 42) comprises a plurality of longitudinal first louvers (46), wherein each of the first louvers (46) has an entry portion (66) facing the fan (22) and at least the entry portion (66) is inclined relative to the center axis (30) of the fan (22), wherein the angle of inclination (α) changes in the longitudinal direction of the respective first louver (46).'

In [6]:


# Download OpenAlex dataset
openalex_df = pd.read_csv(
    "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_openalex.csv"
)

# Download Patents dataset
patents_df = pd.read_json(
    "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_patents.json",
    lines=True,
)

# Optional: Save locally for offline reuse
openalex_df.to_csv("heat_pumps_openalex.csv", index=False)
patents_df.to_json("heat_pumps_patents.json", orient="records", lines=True)

print("Datasets downloaded and saved locally.")


/tmp/ipykernel_7240/3393018013.py:2: DtypeWarning: Columns (52) have mixed types. Specify dtype option on import or set low_memory=False.
  openalex_df = pd.read_csv(


Datasets downloaded and saved locally.


In [7]:
import openai
from openai import OpenAI
client = OpenAI()

In [8]:
# 0 ── Imports & basic config
import math
from pathlib import Path
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm
import json
from time import sleep
from dotenv import load_dotenv
import os
from openai import OpenAI
import re

/home/pascualdiego/projects/DiscoveryHP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
#For Circularity Score#
!pip install nltk


import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')  # Optional but improves lemmatizer accuracy


/bin/bash: line 1: pip: command not found


[nltk_data] Downloading package wordnet to
[nltk_data]     /home/pascualdiego/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/pascualdiego/nltk_data...


True

In [13]:
# -------------------------------
# Load environment and OpenAI client
# -------------------------------
load_dotenv()  # Adjust path if needed
client = OpenAI()


In [12]:
# --------------------------------------------------------------------------- #
# Heat Pump Thesis Classifier  | fully integrated with environment + KWS #
# --------------------------------------------------------------------------- #
#  What this notebook does end‑to‑end:                                        #
#   1.  Loads patents (JSONL) and research papers (CSV)                       #
#   2.  Calls OpenAI gpt‑4o‑mini model to                            #
#         • classify each item into one of 22 sub‑categories                  #
#         • extract short abstract + core innovation                          #
#         • assign cost, efficiency, circularity scores (0‑3)                 #

import os, json, math, pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from discovery_heat_pump_futures import PROJECT_DIR
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# Load environment + OpenAI
load_dotenv(dotenv_path=PROJECT_DIR / ".env")
client = OpenAI()

MODEL_NAME = "gpt-4o-mini"
T_CLASSIFY = 0.1
OUTPUT_PATH = PROJECT_DIR / "outputs/fullset_classified_patents_CEscore.jsonl"
MAX_WORKERS = 10
filter_keywords = True  # ← toggle keyword filtering

# ---- 1. CATEGORIES ---------------------------------------------------------
CATEGORIES = {
    "1.1": "Compressors", "1.2": "Refrigerants", "1.3": "Heat-exchangers", "1.4": "Motor & Drives", "1.5": "Lubrication & oil management",
    "2.1": "Elastocaloric", "2.2": "Electrocaloric", "2.3": "Magnetocaloric", "2.4": "Ionocaloric", "2.5": "Barocaloric",
    "2.6": "Thermoelectric", "2.7": "Electro- & Chemisorption", "2.8": "Thermoacoustic", "2.9": "Sorption / Absorption", "2.10": "Hybrid & cascade",
    "3.1": "Topology & configuration", "3.2": "Controls & optimisation", "3.3": "Operational integration",
    "4.1": "Flexible cycles", "4.2": "Defrost & icing mitigation", "4.3": "Thermal storage",
    "5.1": "Design-for-disassembly", "5.2": "Modular assemblies", "5.3": "Recycled materials", "5.4": "Additive manufacturing", "5.5": "Predictive maintenance"
}

# ---- 2. KEYWORD UNIVERSE ---------------------------------------------------
KWS = {
    "1.1": ["scroll", "rotary", "vane", "twin-screw", "reciprocating", "isothermal compressor", "oil-free", "variable-speed", "economiser"],
    "1.2": ["R32", "R454B", "R290", "propane", "CO₂", "carbon dioxide", "low-GWP", "natural refrigerant", "azeotrope", "zeotropic", "ionic liquid"],
    "1.3": ["micro-channel", "plate heat exchanger", "fin-tube", "enthalpy exchanger", "phase-change heat exchanger", "anti-fouling", "frost-free", "3-D printed"],
    "1.4": ["IPM motor", "SiC inverter", "PMSM", "sensor-less", "flux-weakening"],
    "1.5": ["oil-separator", "mist injection", "CRII", "low-viscosity oil"],
    "2.1": ["elastocaloric", "shape-memory", "Ni-Ti"], "2.2": ["electrocaloric", "ferroelectric"], "2.3": ["magnetocaloric", "gadolinium"],
    "2.4": ["ionocaloric", "ion-solvation"], "2.5": ["barocaloric"], "2.6": ["thermoelectric", "Peltier", "Seebeck"],
    "2.7": ["electrochemical compressor", "chemisorption"], "2.8": ["thermoacoustic"],
    "2.9": ["adsorption heat pump", "absorption heat pump", "lithium bromide"], "2.10": ["cascade heat pump", "hybrid heat pump"],
    "3.1": ["cascade", "booster", "trans-critical", "bi-valent", "ground-source"], "3.2": ["model predictive control", "MPC", "PID", "fault detection", "digital twin"],
    "3.3": ["smart-grid", "thermal district network", "hybrid boiler"], "4.1": ["ejector cycle", "regenerative cycle", "parallel-compressor"],
    "4.2": ["defrost", "icing sensor", "hot-gas bypass", "nano-coating"], "4.3": ["PCM storage", "phase change material", "heat battery", "stratified tank"],
    "5.1": ["tool-less", "snap-fit", "reversible adhesive", "design-for-disassembly", "DfD"], "5.2": ["modular cartridge", "remanufactur", "refurbish", "life-extension"],
    "5.3": ["recycled", "bio-polymer", "PCR plastic", "reclaimed copper", "low-GWP foam"], "5.4": ["additive manufacturing", "3-D print", "near-net shape", "binder-jet"],
    "5.5": ["predictive maintenance", "condition monitoring", "service-as-a-product"]
}
KW2CAT = {kw.lower(): cat for cat, kwlist in KWS.items() for kw in kwlist}

# ---- 3. ISO HEURISTIC ------------------------------------------------------
ISO_LEVERS = {
    "maintain": [
        "predictive maintenance", "preventive maintenance", "condition monitoring",
        "remote diagnostics", "fault detection", "service-as-a-product", "self-healing system"
    ],
    "reuse": [
        "remanufacture", "remanufacturing", "refurbish", "refurbished", "repair", "repairability",
        "cartridge", "replaceable module", "core exchange", "component reuse", "second life"
    ],
    "recycle": [
        "recycled", "recycling", "reclaim", "reclaimed", "material recovery",
        "mono-material", "material loop", "mechanical recycling", "chemical recycling", "closed-loop"
    ],
    "reduce": [
        "near-net shape", "additive manufacturing", "3d printing", "lightweight design",
        "material efficiency", "yield improvement", "low-waste", "minimal material use", "net-shape forming"
    ],
    "regenerate": [
        "bio-based", "biobased", "bio-polymer", "biopolymer", "biocomposite",
        "renewable feedstock", "natural material", "biodegradable", "biomaterial", "plant-derived"
    ]
}

import re
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def normalize(text):
    # Lowercase, strip punctuation, lemmatize
    words = re.findall(r'\b\w+\b', text.lower())
    return [lemmatizer.lemmatize(w) for w in words]

def iso_score(text):
    hits = []
    tokens = normalize(text)
    token_text = " ".join(tokens)  # faster for substring matching

    for kws in ISO_LEVERS.values():
        matched = sum(1 for kw in kws if any(kw in token_text for kw in normalize(kw)))
        hits.append(0 if matched == 0 else 1 if matched == 1 else 2 if matched == 2 else 3)
    return math.ceil(sum(hits) / 5)


# ---- 4. PROMPT GENERATOR ---------------------------------------------------
def make_prompt(title, abstract):
    return f"""
You are a sustainable heating technology analyst.

TITLE: {title}
ABSTRACT: {abstract}

Answer:
1. What is the core technical innovation? (max 25 words)
2. Is it relevant to heat pumps? ('yes' or 'no')
3. If yes, assign exactly ONE category code: {', '.join(CATEGORIES.keys())}
4. Score 0-3 for:
   a) cost_reduction_potential
   b) efficiency_gain_potential
   c) circularity_score (use ISO 59004 lever heuristic)
 Respond ONLY as JSON:
{{"is_relevant": "...", "summary": "...", "category": "...",
  "cost_score": ..., "efficiency_score": ..., "circularity_score": ...}}
""".strip()

# ---- 5. RUN PARALLEL CLASSIFICATION ----------------------------------------
print("▶︎ Loading and preparing data ...")
patents_df = pd.read_json("heat_pumps_patents.json", lines=True)
print("Total patents loaded:", len(patents_df))

if filter_keywords:
    patents_df["text"] = (patents_df["title"] + " " + patents_df["abstract"]).str.lower()
    patents_df = patents_df[patents_df["text"].apply(lambda t: any(k in t for k in KW2CAT))].copy()
    print("After keyword filtering:", len(patents_df))

# Load already processed IDs
processed_ids = set()
if OUTPUT_PATH.exists():
    with open(OUTPUT_PATH, "r") as f:
        processed_ids = {json.loads(line)["publication_number"] for line in f}
    print("Resuming from checkpoint. Already done:", len(processed_ids))

# Classifier function
def classify(row):
    if row["publication_number"] in processed_ids:
        return None
    try:
        prompt = make_prompt(row["title"], row["abstract"])
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=T_CLASSIFY,
            response_format={"type": "json_object"}
        )
        result = json.loads(response.choices[0].message.content)

        # Add both LLM and ISO circularity scores
        combined_text = f"{row['title']} {row['abstract']}"
        result["circularity_score_llm"] = result.pop("circularity_score")
        result["circularity_score_iso"] = iso_score(combined_text)

        result.update({
            "publication_number": row.get("publication_number", ""),
            "title": row["title"],
            "abstract": row["abstract"],
            "url": row.get("url", "")
        })
        return result
    except Exception as e:
        print(f"⚠️ Error on {row.get('publication_number', '')}: {e}")
        return None



# Run it
print("▶︎ Starting classification ...")
t0 = datetime.now()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor, open(OUTPUT_PATH, "a") as f_out:
    futures = [executor.submit(classify, row) for _, row in patents_df.iterrows()]
    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        if result:
            f_out.write(json.dumps(result) + "\n")
print(f"🎉 Done in {datetime.now() - t0}")
print(f"✅ Output saved to: {OUTPUT_PATH}")

▶︎ Loading and preparing data ...
Total patents loaded: 45957
After keyword filtering: 9304
▶︎ Starting classification ...


  0%|          | 0/9304 [00:00<?, ?it/s]

2025-05-20 16:35:55,531 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-05-20 16:35:55,616 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-05-20 16:35:55,734 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-05-20 16:35:55,783 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-05-20 16:35:55,811 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-05-20 16:35:55,877 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-05-20 16:35:56,792 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-05-20 16:35:56,815 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-05-20 16:35:56,829 - httpx 

  0%|          | 2/9304 [00:20<21:38:56,  8.38s/it]

⚠️ Error on US-10753655-B2: 'WordNetCorpusReader' object has no attribute '_LazyCorpusLoader__args'
2025-05-20 16:36:16,704 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

2025-05-20 17:03:48,984 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-05-20 17:03:49,247 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


100%|█████████▉| 9302/9304 [27:54<00:00,  4.99it/s]

2025-05-20 17:03:49,338 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-05-20 17:03:50,022 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


100%|██████████| 9304/9304 [27:58<00:00,  5.54it/s]


🎉 Done in 0:27:59.656706
✅ Output saved to: /home/pascualdiego/projects/DiscoveryHP/outputs/fullset_classified_patents_CEscore.jsonl


In [ ]:
# --------------------------------------------------------------------------- #
# 1 and 2. CLASSIFY OPENALEX RESEARCH PAPERS (Same pipeline as patents)             #
# --------------------------------------------------------------------------- #

OPENALEX_PATH = "heat_pumps_openalex.csv"
OPENALEX_OUTPUT = PROJECT_DIR / "outputs/fullset_classified_papers_CEscore.jsonl"

print("▶︎ Loading OpenAlex research dataset ...")
openalex_df = pd.read_csv(OPENALEX_PATH)
print("Total papers loaded:", len(openalex_df))

# Optional: filter rows by keyword if enabled
if filter_keywords:
    openalex_df["text"] = (
        openalex_df["title"].fillna('') + " " + openalex_df["abstract"].fillna('')
    ).str.lower()

    openalex_df = openalex_df[openalex_df["text"].str.strip() != ""]

    openalex_df = openalex_df[
        openalex_df["text"].apply(lambda t: any(k in t for k in KW2CAT))
    ].copy()

    print("After keyword filtering:", len(openalex_df))

# Load already processed DOIs
processed_dois = set()
if OPENALEX_OUTPUT.exists():
    with open(OPENALEX_OUTPUT, "r") as f:
        processed_dois = {json.loads(line).get("doi_url", "") for line in f}
    print("Resuming from checkpoint. Already done:", len(processed_dois))

# Build prompt for research paper
def make_prompt_paper(title, abstract):
    return f"""
You are a sustainable heating technology analyst.

TITLE: {title}
ABSTRACT: {abstract}

Answer:
1. What is the core technical innovation? (max 25 words)
2. Is it relevant to heat pumps? ('yes' or 'no')
3. If yes, assign exactly ONE category code: {', '.join(CATEGORIES.keys())}
4. Score 0-3 for:
   a) cost_reduction_potential
   b) efficiency_gain_potential
   c) circularity_score (use ISO 59004 lever heuristic)
   Respond ONLY as JSON:
{{"is_relevant": "...", "summary": "...", "category": "...",
  "cost_score": ..., "efficiency_score": ..., "circularity_score": ...}}
""".strip()

# LLM wrapper
def classify_paper(row):
    doi = row.get("doi_url", "")
    if doi in processed_dois:
        return None
    try:
        prompt = make_prompt_paper(row["title"], row["abstract"])
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=T_CLASSIFY,
            response_format={"type": "json_object"}
        )
        result = json.loads(response.choices[0].message.content)

        # Add both LLM and ISO circularity scores
        combined_text = f"{row['title']} {row['abstract']}"
        result["circularity_score_llm"] = result.pop("circularity_score")
        result["circularity_score_iso"] = iso_score(combined_text)

        result.update({
            "title": row["title"],
            "abstract": row["abstract"],
            "doi_url": doi
        })
        return result
    except Exception as e:
        print(f"⚠️ Error on paper: {e}")
        return None



# Run in parallel
print("▶︎ Starting OpenAlex classification ...")
t0 = datetime.now()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor, open(OPENALEX_OUTPUT, "a") as f_out:
    futures = [executor.submit(classify_paper, row) for _, row in openalex_df.iterrows()]
    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        if result:
            f_out.write(json.dumps(result) + "\n")
print(f"🎉 Done in {datetime.now() - t0}")
print(f"✅ Output saved to: {OPENALEX_OUTPUT}")
